# Week 16, Day 6: Forge: Week 16 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w016/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
odds = 10 / 99
print("1. posterior =", round(odds / (1 + odds), 4))
print("2. weight =", (1 / 1 ** 2) / (1 / 1 ** 2 + 1 / 3 ** 2))
print("3. factor sqrt(1 - 0.8^2) =", np.sqrt(1 - 0.8 ** 2))

### Exercise: Part A: calibration and certainty

For 2,000 trials, compute the posterior on `grid`, its 68% credible set (the most probable grid points until their probability reaches 0.68), whether that set contains the truth (within half a grid step), the posterior standard deviation around the MAP, and the total spike count. Store `coverage`, `post_sd` and `total_spikes`.

In [ ]:
rng = np.random.default_rng(2006)
grid = np.arange(0, 180, 0.5)
def circ(d): d = np.mod(d, 180); return np.minimum(d, 180 - d)
N = 16
prefs = np.arange(N) * 180 / N
def tuning(theta, p): return 0.5 + 6 * np.exp(2 * (np.cos(np.deg2rad(2 * np.subtract.outer(theta, p))) - 1))
F = tuning(grid, prefs)                                       # rates for every grid orientation (gain 1)

hits, post_sd, total_spikes = [], [], []
for _ in range(2000):
    theta, gain = rng.uniform(0, 180), rng.uniform(0.5, 2.0)
    k = rng.poisson(gain * tuning(np.array([theta]), prefs)[0])
    loglik = (k * np.log(F) - F).sum(axis=1)                 # the decoder assumes gain 1
    p = np.exp(loglik - loglik.max()); p /= p.sum()
    order = np.argsort(-p)
    credible = order[: np.searchsorted(np.cumsum(p[order]), 0.68) + 1]
    hits.append(...)                                          # does the credible set contain theta?
    m = grid[np.argmax(p)]
    d = np.mod(grid - m + 90, 180) - 90                       # signed distance from the MAP
    post_sd.append(...)
    total_spikes.append(k.sum())
coverage = np.mean(hits)
print(f"coverage of 68% credible sets: {coverage:.3f};  correlation of total spikes with posterior sd: {np.corrcoef(total_spikes, post_sd)[0, 1]:.2f}")
plt.plot(total_spikes, post_sd, ".", alpha=0.3); plt.xlabel("total spikes on the trial"); plt.ylabel("posterior sd (degrees)"); plt.show()

In [ ]:
import numpy as np
def test_coverage():
    "the 68% credible sets contain the true orientation on 63% to 73% of trials"
    assert 0.63 < coverage < 0.73

def test_spikes_certainty():
    "trials with more spikes give narrower posteriors: correlation below -0.6"
    assert np.corrcoef(total_spikes, post_sd)[0, 1] < -0.6

test_coverage()
test_spikes_certainty()
print("All checks passed")

In [ ]:
true = 55.0
def observer(sV, sH, n, conflict=0.0, mode="both"):
    xV = true - conflict / 2 + sV * rng.normal(size=n)
    xH = true + conflict / 2 + sH * rng.normal(size=n)
    if mode == "vision": return xV
    if mode == "touch": return xH
    wopt = (1 / sV ** 2) / (1 / sV ** 2 + 1 / sH ** 2)
    w = min(1.0, wopt + 0.15)                       # this observer leans on vision a little too much
    return w * xV + (1 - w) * xH
blur_levels = {"sharp": 1.0, "blurred": 2.5}
sigma_H_true = 1.8

### Exercise: Part B: weights and precision

Use 3,000 trials of each kind. Estimate `sH_hat` from touch-alone trials and `sV_hat[level]` from vision-alone trials. For each blur level, predict the optimal visual weight `w_pred[level]`, measure the observer's weight `w_meas[level]` from conflict trials, and record the sd of no-conflict combined trials in `sd_comb[level]`.

In [ ]:
n = 3000
sH_hat = observer(1.0, sigma_H_true, n, mode="touch").std()
sV_hat, w_pred, w_meas, sd_comb = {}, {}, {}, {}
for level, sV in blur_levels.items():
    sV_hat[level] = observer(sV, sigma_H_true, n, mode="vision").std()
    w_pred[level] = ...
    est_conflict = observer(sV, sigma_H_true, n, conflict=3.0)
    w_meas[level] = ...                                     # (true + 1.5 - mean) / 3, as on Day 3
    sd_comb[level] = observer(sV, sigma_H_true, n).std()
    best = min(sV_hat[level], sH_hat)
    print(f"{level:8s} predicted weight {w_pred[level]:.2f}, measured {w_meas[level]:.2f};  combined sd {sd_comb[level]:.2f}, better cue {best:.2f}")

In [ ]:
import numpy as np
def test_noise_estimates():
    "single-cue noise estimates are within 5% of the observer's true noise"
    assert abs(sH_hat - 1.8) / 1.8 < 0.05 and all(abs(sV_hat[k] - v) / v < 0.05 for k, v in blur_levels.items())

def test_overweight():
    "at both blur levels the measured visual weight exceeds the predicted optimal weight by more than 0.08"
    assert all(w_meas[k] - w_pred[k] > 0.08 for k in blur_levels)

def test_precision():
    "even so, the combined sd beats the better single cue at both blur levels"
    assert all(sd_comb[k] < min(sV_hat[k], sH_hat) for k in blur_levels)

test_noise_estimates()
test_overweight()
test_precision()
print("All checks passed")

### Exercise: Part C: Gaussian conditioning on data

Write `predict0(train, test)` that estimates the mean and covariance from `train`, then returns the conditional mean of neuron 0 given neurons 1 to 19 for each test trial. Compute the test R² (1 − mean squared error / variance) for 30 and 200 training trials, and `r2_true` using the true covariance.

In [ ]:
n_neur, k = 20, 3
Lf = rng.normal(0, 1, (n_neur, k))
private = rng.uniform(0.5, 1.5, n_neur); private[0] = 0.3          # neuron 0 has little private noise
S_true = Lf @ Lf.T + np.diag(private)
data = rng.multivariate_normal(np.zeros(n_neur), S_true, 2200)
train_all, test = data[:200], data[200:]
o = np.arange(1, n_neur)

def predict0(train, test):
    mu = train.mean(0)
    C = np.cov(train.T)
    w = ...                                         # C_oo^-1 C_o0
    return ...

def R2(pred): return 1 - np.mean((test[:, 0] - pred) ** 2) / np.var(test[:, 0])
r2 = {m: R2(predict0(train_all[:m], test)) for m in (30, 200)}
w_true = np.linalg.solve(S_true[np.ix_(o, o)], S_true[o, 0])
r2_true = R2(test[:, o] @ w_true)
print({m: round(v, 3) for m, v in r2.items()}, "  true covariance:", round(r2_true, 3))

In [ ]:
import numpy as np
def test_r2_good():
    "with 200 training trials the prediction explains more than 80% of neuron 0's variance on test trials"
    assert r2[200] > 0.8

def test_more_data():
    "more training trials give a better prediction, approaching the true-covariance limit"
    assert r2[30] < r2[200] <= r2_true + 0.02

test_r2_good()
test_more_data()
print("All checks passed")